[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kauelopes/topicos-ml/blob/main/aulas/01-python-numpy-jax/notebooks/01_demo.ipynb)

# Aula 1 — Demonstração: Python, NumPy e JAX
*Computação Aplicada à Economia: Tópicos em Aprendizado de Máquina*

Roteiro do live coding:
1. Python essencial: tipos, coleções, `if`, `for`, `def`
2. NumPy: do laço à operação vetorizada; MQO matricial
3. JAX: `grad`, `jit`, `vmap`

## 1. Python essencial

In [ ]:
selic = 15.0          # float
n_meses = 12          # int
pais = "Brasil"       # str
acima_da_meta = True  # bool

taxa_mensal = (1 + selic / 100) ** (1 / n_meses) - 1
print(f"Selic mensal equivalente: {taxa_mensal:.4%}")
type(selic), type(n_meses)

In [ ]:
# Ponto flutuante não é exato
import math
print(0.1 + 0.2 == 0.3, math.isclose(0.1 + 0.2, 0.3))

In [ ]:
ipca = [0.16, 0.43, 0.56, 0.43]       # variação mensal (%)
metas = {"centro": 3.0, "tolerancia": 1.5}
print(ipca[0], ipca[-1], ipca[1:3])
print("Teto:", metas["centro"] + metas["tolerancia"])

In [ ]:
def acumulada(taxas_pct):
    """Inflação acumulada (%) a partir de taxas mensais (%)."""
    fator = 1.0
    for pi in taxas_pct:
        fator = fator * (1 + pi / 100)
    return 100 * (fator - 1)


def avaliar(inflacao_12m, centro=3.0, tolerancia=1.5):
    if inflacao_12m > centro + tolerancia:
        return "acima do teto"
    elif inflacao_12m < centro - tolerancia:
        return "abaixo do piso"
    return "dentro do intervalo"


acumulada(ipca), avaliar(5.2), avaliar(2.0)

## 2. NumPy: computação vetorizada

In [ ]:
import numpy as np

pi = np.array(ipca)
X = np.array([[1, 2.0], [1, 3.5], [1, 1.2]])
print(pi.shape, X.shape)
print(X[:, 1])
print(X[X[:, 1] > 1.5])

### Laço × vetorizado

In [ ]:
rng = np.random.default_rng(0)
taxas = rng.normal(0.4, 0.3, size=1_000_000)

%timeit acumulada(taxas)
%timeit 100 * (np.prod(1 + taxas / 100) - 1)

### Broadcasting: padronizar colunas

In [ ]:
X_pad = (X - X.mean(axis=0)) / np.where(X.std(axis=0) > 0, X.std(axis=0), 1)
X.mean(axis=0).shape, X_pad

### MQO em uma linha
Equações normais: $(X^\top X)\hat\beta = X^\top y$. **Resolva o sistema**, não inverta a matriz.

In [ ]:
rng = np.random.default_rng(42)
n = 500
x = rng.normal(size=n)
X = np.column_stack([np.ones(n), x])
y = 1.0 + 2.0 * x + rng.normal(size=n)

beta = np.linalg.solve(X.T @ X, X.T @ y)
beta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
beta, beta_lstsq

## 3. JAX

In [ ]:
import jax
import jax.numpy as jnp

x = jnp.arange(5.0)
# x[0] = 10.0          # erro: arrays JAX são imutáveis
x = x.at[0].set(10.0)
x

### `grad`: utilidade marginal CRRA, $u(c)=c^{1-\gamma}/(1-\gamma)$

In [ ]:
gamma = 2.0
def u(c):
    return c ** (1 - gamma) / (1 - gamma)

du = jax.grad(u)
print(du(1.5), 1.5 ** -gamma)

c = jnp.linspace(0.5, 3.0, 6)
jax.vmap(du)(c), c ** -gamma

### Gradiente da perda quadrática
$\mathcal L(\beta) = \frac1n\lVert y - X\beta\rVert^2$, $\quad \nabla_\beta \mathcal L = -\frac2n X^\top(y - X\beta)$

In [ ]:
def perda(beta, X, y):
    r = y - X @ beta
    return jnp.mean(r ** 2)

grad_perda = jax.grad(perda)
b0 = jnp.zeros(2)
print(grad_perda(b0, X, y))
print(-2 / n * X.T @ (y - X @ b0))
print("No MQO:", grad_perda(beta, X, y))   # ~ 0

### `jit`: compilar

In [ ]:
grad_rapido = jax.jit(grad_perda)
grad_rapido(b0, X, y).block_until_ready()   # 1ª chamada: compila

%timeit grad_perda(b0, X, y).block_until_ready()
%timeit grad_rapido(b0, X, y).block_until_ready()